# Seasonal Patterns Analysis

Explore how propagation varies with seasons and the solar cycle.

**What you'll learn:**
- Monthly propagation patterns by band
- Solar cycle effects over multiple years
- Seasonal differences in band behavior

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import load_dataset

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
# Load propagation data
df = load_dataset("wspr")
print(f"Loaded {len(df):,} signatures")

# Load solar data if available
try:
    solar = load_dataset("solar")
    print(f"Solar data: {len(solar):,} days")
except FileNotFoundError:
    solar = None
    print("Solar dataset not available")

## Seasonal Effects on Propagation

**Summer:**
- Sporadic-E openings on 6m and 10m
- D-layer absorption stronger (longer daylight)
- Low band DX difficult (short nights)

**Winter:**
- Better low-band DX (long dark periods)
- Reduced sporadic-E
- High bands still work during day if SFI is adequate

**Equinoxes (March/September):**
- Often best overall HF conditions
- Geomagnetic activity sometimes elevated
- Good balance of day/night paths

In [ ]:
# Monthly SNR by band
BAND_NAMES = {103: "80m", 105: "40m", 107: "20m", 109: "15m", 111: "10m"}
BANDS = list(BAND_NAMES.keys())

fig, ax = plt.subplots(figsize=(14, 7))

for band in BANDS:
    band_df = df[df["band"] == band]
    monthly = band_df.groupby("month")["median_snr"].mean()
    ax.plot(monthly.index, monthly.values, marker="o", 
            label=BAND_NAMES[band], linewidth=2)

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Month")
ax.set_ylabel("Mean SNR (dB)")
ax.set_title("Monthly Propagation Patterns by Band")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.legend(title="Band")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal heatmaps
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, band in enumerate(BANDS):
    band_df = df[df["band"] == band]
    pivot = band_df.groupby(["hour", "month"])["median_snr"].mean().unstack(fill_value=np.nan)
    
    sns.heatmap(pivot, ax=axes[i], cmap="RdYlGn", center=0,
                cbar_kws={"label": "SNR (dB)"})
    axes[i].set_title(f"{BAND_NAMES[band]} Band")
    axes[i].set_xlabel("Month")
    axes[i].set_ylabel("Hour (UTC)")

axes[-1].axis("off")
plt.suptitle("Seasonal Hour×Month Patterns by Band", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Solar cycle visualization (if solar data available)
if solar is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Convert date column if needed
    if "date" in solar.columns:
        solar["date"] = pd.to_datetime(solar["date"])
        solar = solar.set_index("date")
    
    # Monthly averages
    monthly_solar = solar.resample("M").mean()
    
    # SFI over time
    if "sfi" in monthly_solar.columns:
        axes[0].plot(monthly_solar.index, monthly_solar["sfi"], 
                    color="orange", linewidth=1.5)
        axes[0].set_ylabel("Solar Flux Index (SFU)")
        axes[0].set_title("Solar Flux Index Over Time")
        axes[0].grid(True, alpha=0.3)
    
    # SSN over time
    if "ssn" in monthly_solar.columns:
        axes[1].plot(monthly_solar.index, monthly_solar["ssn"],
                    color="blue", linewidth=1.5)
        axes[1].set_ylabel("Sunspot Number")
        axes[1].set_xlabel("Date")
        axes[1].set_title("Sunspot Number Over Time")
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Solar dataset not available for solar cycle plot.")
    print("Download with: ionis-download --datasets solar")

In [ ]:
# SFI correlation by month
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

seasons = {
    "Winter (Dec-Feb)": [12, 1, 2],
    "Spring (Mar-May)": [3, 4, 5],
    "Summer (Jun-Aug)": [6, 7, 8],
    "Fall (Sep-Nov)": [9, 10, 11],
}

for ax, (season_name, months) in zip(axes.flatten(), seasons.items()):
    season_df = df[df["month"].isin(months)].copy()
    season_df["sfi_bracket"] = (season_df["avg_sfi"] // 25) * 25
    
    for band in [107, 109, 111]:  # 20m, 15m, 10m
        band_df = season_df[season_df["band"] == band]
        agg = band_df.groupby("sfi_bracket")["median_snr"].mean()
        if len(agg) > 2:
            ax.plot(agg.index, agg.values, marker="o", 
                   label=BAND_NAMES[band], linewidth=2)
    
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("SFI Bracket")
    ax.set_ylabel("Mean SNR (dB)")
    ax.set_title(season_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Solar Correlation by Season (High Bands)", fontsize=14)
plt.tight_layout()
plt.show()

## Key Insights

**Seasonal patterns:**
- Equinox months (Mar/Sep) often show peak high-band activity
- Low bands peak in winter months (longer darkness)
- Summer sporadic-E creates 10m/6m openings outside normal solar patterns

**Solar cycle effects:**
- High bands (10m, 15m) very dependent on solar activity
- Solar maximum: all bands active, long-distance 10m possible
- Solar minimum: high bands may not open at all, low bands dominate

**Planning implications:**
- Contest scheduling considers seasonal propagation
- DXpeditions time trips to maximize band openings
- Equipment choices vary by expected conditions